# Detection Quality Improvements

The goal of this analysis is to identify which changes improve detection performance the most on unseen videos.

Three types of improvements are compared:

1. Model: Changes to the architecture and the use of different object detection models.
2. Data: Using scale augmentation during training.
3. Inference strategy: Model ensembles, TTA, temporal post-processing, and fusion.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
ROOT = Path('.').resolve().parent

## Model and data changes: single checkpoints, no TTA


In [2]:
def five_fold_rows(name, change, cv, official):
    return dict(model=name, change=change, cv_mAP=float(pd.Series(cv).mean()), official_mAP=float(pd.Series(official).mean()),
                folds=len(cv))


def final_evaluation_rows(model, experiment):
    folds = pd.read_csv(ROOT / model / 'final_evaluation' / f'{model}_folds.csv').query('experiment == @experiment')
    return (folds.query("evaluation_split == 'cv_out_of_fold'")['map'].tolist(),
            folds.query("evaluation_split == 'official_val'")['map'].tolist())


rows = []
faster_v1_cv, faster_v1_official = final_evaluation_rows('faster_rcnn', 'lr_scheduler')
rows.append(five_fold_rows('Faster R-CNN v1', 'reference', faster_v1_cv, faster_v1_official))

In [3]:
v2_no_aug_file = ROOT / 'faster_rcnn/faster_rcnn_v2/v2_lr_scheduler_5fold.csv'
if v2_no_aug_file.exists():
    v2_no_aug = pd.read_csv(v2_no_aug_file)
    rows.append(five_fold_rows('Faster R-CNN v2', 'model: v1 -> v2 architecture', v2_no_aug.cv_mAP, v2_no_aug.official_mAP))
v2_scale_aug = pd.read_csv(ROOT / 'faster_rcnn/faster_rcnn_v2/v2_lr_scheduler_scale_aug_5fold.csv')
rows.append(five_fold_rows('Faster R-CNN v2 + scale augmentation', 'data: + scale augmentation', v2_scale_aug.cv_mAP,
                           v2_scale_aug.official_mAP))

In [4]:
retina = pd.read_csv(ROOT / 'retinanet/five_fold_results.csv')
rows.append(five_fold_rows('RetinaNet', 'model: one-stage anchor-based', retina.mAP, retina.official_mAP))

fcos_cv, fcos_official = final_evaluation_rows('fcos', 'F1_ciou')
rows.append(five_fold_rows('FCOS', 'model: one-stage anchor-free', fcos_cv, fcos_official))

In [5]:
rtdetr = pd.read_csv(ROOT / 'rt_detr/five_fold_results.csv')
for config, change in [('recipe_warmup_ema', 'model: query-based (baseline recipe)'), ('no_denoising', 'model: + no denoising loss'),
                       ('no_denoising_640x1152', 'model/data: + input 640 x 1152')]:
    runs = rtdetr[rtdetr.config == config]
    if runs.fold.nunique() == 5:
        rows.append(five_fold_rows(f'RT-DETR {config}', change, runs.mAP, runs.official_mAP))
single_checkpoints = pd.DataFrame(rows)
display(single_checkpoints.round(4))

,model,change,cv_mAP,official_mAP,folds
0,Faster R-CNN v1,reference,0.2110,0.1909,5
1,Faster R-CNN v2,model: v1 -> v2 architecture,0.2312,0.2109,5
2,Faster R-CNN v2 + scale augmentation,data: + scale augmentation,0.2437,0.2219,5
3,RetinaNet,model: one-stage anchor-based,0.1787,0.1483,5
4,FCOS,model: one-stage anchor-free,0.1969,0.1793,5
5,RT-DETR recipe_warmup_ema,model: query-based (baseline recipe),0.1691,0.1485,5
6,RT-DETR no_denoising,model: + no denoising loss,0.2258,0.2147,5
7,RT-DETR no_denoising_640x1152,model/data: + input 640 x 1152,0.2602,0.2360,5


## Inference strategies on fixed models (official validation)

In [6]:
ablation = pd.read_csv(ROOT / 'quality_improvements/pipeline_ablation_official_val.csv')
display(ablation[['model', 'step', 'mAP', 'AP50', 'APs', 'AR']].round(4))

,model,step,mAP,AP50,APs,AR
0,v1,1 checkpoint (mean of 5),0.1909,0.3937,0.0879,0.3232
1,v1,"5 checkpoints, WBF, no TTA",0.2153,0.4384,0.1060,0.3848
2,v1,"5 checkpoints x 6 TTA views, WBF",0.2364,0.4771,0.1267,0.4727
3,v1,+ temporal processing,0.2450,0.4987,0.1315,0.4731
4,v1,+ FCOS fusion + temporal,0.2528,0.5044,0.1405,0.4959
5,v2,1 checkpoint (mean of 5),0.2219,0.4466,0.0958,0.4093
6,v2,"5 checkpoints, WBF, no TTA",0.2654,0.5175,0.1262,0.4888
7,v2,"5 checkpoints x 6 TTA views, WBF",0.2717,0.5299,0.1334,0.4994
8,v2,+ temporal processing,0.2841,0.5573,0.1560,0.4996
9,v2,+ FCOS fusion + temporal,0.2928,0.5588,0.1660,0.5113


In [7]:
steps = ablation.pivot_table(index='step', columns='model', values='mAP', sort=False)
steps_gain = steps.diff()
display(pd.concat({'mAP': steps, 'gain of the step': steps_gain}, axis=1).round(4))

mAP         gain of the step        
model                                 v1      v2               v1      v2
step                                                                     
1 checkpoint (mean of 5)          0.1909  0.2219              NaN     NaN
5 checkpoints, WBF, no TTA        0.2153  0.2654           0.0245  0.0435
5 checkpoints x 6 TTA views, WBF  0.2364  0.2717           0.0210  0.0064
+ temporal processing             0.2450  0.2841           0.0086  0.0124
+ FCOS fusion + temporal          0.2528  0.2928           0.0078  0.0087

In [8]:
video_columns = [c for c in ablation.columns if c.startswith('mAP_video_')]
per_video = ablation.set_index(['model', 'step'])[video_columns]
per_video.columns = [c.replace('mAP_video_', '') for c in video_columns]
display(per_video.round(4))

uav0000086_00000_v  uav0000117_02622_v  uav0000137_00458_v  uav0000182_00000_v  uav0000268_05773_v  uav0000305_00000_v  uav0000339_00001_v
model step                                                                                                                                                                        
v1    1 checkpoint (mean of 5)                      0.1461              0.2344              0.2628              0.1608              0.1085              0.2342              0.2420
      5 checkpoints, WBF, no TTA                    0.1690              0.2671              0.2892              0.1832              0.1292              0.2756              0.2636
      5 checkpoints x 6 TTA views, WBF              0.1610              0.2915              0.3066              0.2037              0.1810              0.3194              0.2785
      + temporal processing                         0.1787              0.3037              0.3156              0.2144              0.1844              0.3258              0.2969
      + FCOS fusion + temporal                      0.1926              0.3006              0.3200              0.2197              0.1881              0.3699              0.2947
v2    1 checkpoint (mean of 5)                      0.2272              0.2564              0.3077              0.1868              0.1202              0.2649              0.2584
      5 checkpoints, WBF, no TTA                    0.3044              0.2999              0.3493              0.2248              0.1505              0.3303              0.2983
      5 checkpoints x 6 TTA views, WBF              0.3067              0.3082              0.3573              0.2295              0.1571              0.3369              0.2987
      + temporal processing                         0.3185              0.3279              0.3603              0.2384              0.1612              0.3553              0.3187
      + FCOS fusion + temporal                      0.3034              0.3357              0.3588              0.2415              0.1922              0.3808              0.3404

RT-DETR (`no_denoising_640x1152`, 5 fold checkpoints): the same steps with 8 TTA views (horizontal flip × scales 0.75–1.5),
class-wise NMS 0.5 inside each view and WBF 0.6 over all checkpoint × view sets.

In [9]:
rtdetr_official = pd.read_csv(ROOT / 'rt_detr/inference/official_validation_results.csv').set_index('key')
rtdetr_steps = rtdetr_official.loc[['RT-DETR 5 checkpoints, single view (trained setting)', 'RT-DETR 5 checkpoints TTA + WBF',
                                    'RT-DETR 5-fold TTA + WBF + temporal (tuned)'], ['mAP', 'AP50', 'APs', 'AR']]
rtdetr_steps.index = ['1 checkpoint (mean of 5)', '5 checkpoints x 8 TTA views, WBF', '+ temporal processing']
rtdetr_steps['gain of the step'] = rtdetr_steps['mAP'].diff()
display(rtdetr_steps.round(4))

,mAP,AP50,APs,AR,gain of the step
1 checkpoint (mean of 5),0.2686,0.5134,0.1336,0.5019,NaN
"5 checkpoints x 8 TTA views, WBF",0.2882,0.5437,0.1425,0.5259,0.0196
+ temporal processing,0.3012,0.5706,0.1461,0.5263,0.0129


## Fusion of different detectors



In [10]:
fusion_decisions = {}
v2_ensemble_results_file = ROOT / 'quality_improvements/faster_rcnn_v2_ensemble/official_validation_results.csv'
retina_decision = json.loads((ROOT / 'retinanet/inference/retina_v2_fusion_decision.json').read_text())
fusion_decisions['RetinaNet + v2 + temporal vs v2 + temporal'] = {k: v for k, v in retina_decision.items() if k != 'per_video'}
rtdetr_report = ROOT / 'kaggle_submissions/rtdetr_tta_wbf_temporal_submission_report.json'
if rtdetr_report.exists():
    fusion_decisions['RT-DETR + v2/FCOS ensemble'] = json.loads(rtdetr_report.read_text())['decision_ensemble_fusion']
display(pd.DataFrame(fusion_decisions).T)

,mAP,reference_A,gain,videos_improved,videos,adopt,reference_B
RetinaNet + v2 + temporal vs v2 + temporal,0.284206,0.284104,0.000102,4,7,False,NaN
RT-DETR + v2/FCOS ensemble,0.30512,NaN,0.01231,4,7,False,0.29281


In [11]:
v2_official = pd.read_csv(v2_ensemble_results_file)
v2_official['key'] = v2_official['key'].str.replace(r' \((?:0\.\d+ pipeline|Kaggle 0\.\d+)\)', '', regex=True)
display(v2_official[['key', 'mAP', 'AP50', 'APs', 'AR']].round(4))

,key,mAP,AP50,APs,AR
0,v1: Faster R-CNN 5 checkpoints TTA,0.2364,0.4771,0.1267,0.4727
1,v1: Faster R-CNN + temporal,0.2450,0.4987,0.1315,0.4731
2,v1: Faster R-CNN + FCOS fusion,0.2446,0.4854,0.1280,0.4955
3,v1: Faster R-CNN + FCOS fusion + temporal (tuned),0.2528,0.5044,0.1405,0.4959
4,v2 5 checkpoints TTA,0.2717,0.5299,0.1334,0.4994
5,v2 + temporal (Faster R-CNN parameters),0.2837,0.5546,0.1542,0.4996
6,v2 + FCOS fusion,0.2748,0.5285,0.1378,0.5109
7,A: v2 + temporal (tuned),0.2841,0.5573,0.1560,0.4996
8,B: v2 + FCOS fusion + temporal (tuned),0.2928,0.5588,0.1660,0.5113


## Temporal post-processing across detectors (official validation)

In [12]:
def official_map(csv_file, key):
    table = pd.read_csv(ROOT / csv_file).set_index('key')
    return float(table.loc[key, 'mAP']) if key in table.index else None

temporal_rows = [
    ('Faster R-CNN v1', 'fcos/inference/official_validation_results.csv',
     'Faster R-CNN 5 checkpoints TTA (0.2587 pipeline)', 'Faster R-CNN + temporal (0.2810 pipeline)'),
    ('Faster R-CNN v2 + scale aug', 'quality_improvements/faster_rcnn_v2_ensemble/official_validation_results.csv',
     'v2 5 checkpoints TTA', 'A: v2 + temporal (tuned)'),
    ('FCOS', 'fcos/inference/official_validation_results.csv',
     'FCOS 5 checkpoints, optimized TTA', 'FCOS 5 checkpoints, optimized TTA + temporal (tuned for FCOS)'),
    ('RetinaNet', 'retinanet/inference/official_validation_results.csv',
     'RetinaNet 5 checkpoints TTA + WBF', 'RetinaNet 5-fold TTA + WBF + temporal (tuned)'),
    ('RT-DETR', 'rt_detr/inference/official_validation_results.csv',
     'RT-DETR 5 checkpoints TTA + WBF', 'RT-DETR 5-fold TTA + WBF + temporal (tuned)'),
]

In [13]:
rows = []
for detector, csv_file, before_key, after_key in temporal_rows:
    if not (ROOT / csv_file).exists():
        continue
    before, after = official_map(csv_file, before_key), official_map(csv_file, after_key)
    rows.append(dict(detector=detector, without_temporal=before, with_temporal=after,
                     gain=None if before is None or after is None else after - before))
display(pd.DataFrame(rows).round(4))

,detector,without_temporal,with_temporal,gain
0,Faster R-CNN v1,0.2364,0.2450,0.0086
1,Faster R-CNN v2 + scale aug,0.2717,0.2841,0.0124
2,FCOS,0.2216,0.2367,0.0150
3,RetinaNet,0.2249,0.2419,0.0170
4,RT-DETR,0.2882,0.3012,0.0129


## Best validated pipelines

In [14]:
results = pd.concat([
    pd.read_csv(ROOT / 'quality_improvements/faster_rcnn_v2_ensemble/official_validation_results.csv'),
    pd.read_csv(ROOT / 'rt_detr/inference/official_validation_results.csv'),
]).drop_duplicates('key').set_index('key')
best_pipelines = {
    'Faster R-CNN v1 + FCOS fusion + temporal': 'v1: Faster R-CNN + FCOS fusion + temporal (tuned)',
    'Faster R-CNN v2 + temporal': 'A: v2 + temporal (tuned)',
    'Faster R-CNN v2 + FCOS fusion + temporal': 'B: v2 + FCOS fusion + temporal (tuned)',
    'RT-DETR + temporal': 'RT-DETR 5-fold TTA + WBF + temporal (tuned)',
    'RT-DETR + Faster R-CNN v2 / FCOS fusion + temporal': 'RT-DETR + v2/FCOS ensemble fusion + temporal (tuned)',
}
best = results.loc[list(best_pipelines.values())]
best.index = list(best_pipelines)
display(best[['mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR', 'ms_per_frame']].round(4))

,mAP,AP50,AP75,APs,APm,APl,AR,ms_per_frame
Faster R-CNN v1 + FCOS fusion + temporal,0.2528,0.5044,0.2219,0.1405,0.2392,0.4230,0.4959,18.7449
Faster R-CNN v2 + temporal,0.2841,0.5573,0.2545,0.1560,0.2584,0.4894,0.4996,3.6468
Faster R-CNN v2 + FCOS fusion + temporal,0.2928,0.5588,0.2692,0.1660,0.2659,0.5005,0.5113,16.8026
RT-DETR + temporal,0.3012,0.5706,0.2766,0.1461,0.2751,0.4986,0.5263,6.4748
RT-DETR + Faster R-CNN v2 / FCOS fusion + temporal,0.3051,0.5771,0.2825,0.1578,0.2834,0.5026,0.5306,26.4414


In [15]:
per_video = best[[column for column in best.columns if column.startswith('mAP_video_')]]
per_video.columns = [column.replace('mAP_video_', '') for column in per_video.columns]
display(per_video.round(4))
pairs = [('RT-DETR + temporal', 'Faster R-CNN v2 + temporal'),
         ('RT-DETR + temporal', 'Faster R-CNN v2 + FCOS fusion + temporal'),
         ('RT-DETR + Faster R-CNN v2 / FCOS fusion + temporal', 'Faster R-CNN v2 + FCOS fusion + temporal'),
         ('RT-DETR + Faster R-CNN v2 / FCOS fusion + temporal', 'RT-DETR + temporal')]
display(pd.DataFrame([dict(pipeline=first, reference=second, mAP_difference=best.loc[first, 'mAP'] - best.loc[second, 'mAP'],
                           videos_better=f"{int((per_video.loc[first] > per_video.loc[second]).sum())} / {per_video.shape[1]}")
                      for first, second in pairs]).round(4))

,uav0000086_00000_v,uav0000117_02622_v,uav0000137_00458_v,uav0000182_00000_v,uav0000268_05773_v,uav0000305_00000_v,uav0000339_00001_v
Faster R-CNN v1 + FCOS fusion + temporal,0.1926,0.3006,0.3200,0.2197,0.1881,0.3699,0.2947
Faster R-CNN v2 + temporal,0.3185,0.3279,0.3603,0.2384,0.1612,0.3553,0.3187
Faster R-CNN v2 + FCOS fusion + temporal,0.3034,0.3357,0.3588,0.2415,0.1922,0.3808,0.3404
RT-DETR + temporal,0.4096,0.3287,0.3954,0.2295,0.1723,0.4882,0.3164
RT-DETR + Faster R-CNN v2 / FCOS fusion + temporal,0.3952,0.3347,0.3907,0.2422,0.1873,0.4669,0.3300


,pipeline,reference,mAP_difference,videos_better
0,RT-DETR + temporal,Faster R-CNN v2 + temporal,0.0170,5 / 7
1,RT-DETR + temporal,Faster R-CNN v2 + FCOS fusion + temporal,0.0083,3 / 7
2,RT-DETR + Faster R-CNN v2 / FCOS fusion + temp...,Faster R-CNN v2 + FCOS fusion + temporal,0.0123,4 / 7
3,RT-DETR + Faster R-CNN v2 / FCOS fusion + temp...,RT-DETR + temporal,0.0040,4 / 7
